# 📸 Preprocesar Imágenes Propias
## Convertir fotos de dígitos al formato MNIST (28x28 grayscale)

### Instrucciones:
1. Dibuja dígitos (0-9) con **marcador/pluma oscura en papel blanco**
2. Toma una foto de cada dígito (buena iluminación, de frente)
3. Coloca las fotos en `mis_imagenes/raw/`
4. **Nombra cada archivo como**: `{digito}_{numero}.jpg`
   - Ejemplo: `3_01.jpg`, `3_02.jpg`, `7_01.jpg`, `0_05.jpg`
   - El primer número es el dígito real que dibujaste
5. Ejecuta este notebook para preprocesarlas

---
## 1. Importar Librerías

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageFilter
import os
import csv
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 14

# Rutas
RAW_DIR = Path('../mis_imagenes/raw')
PROC_DIR = Path('../mis_imagenes_procesadas')
PROC_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Librerías importadas')
print(f'   Directorio de fotos: {RAW_DIR.resolve()}')
print(f'   Directorio de salida: {PROC_DIR.resolve()}')

---
## 2. Listar imágenes disponibles

In [ ]:
# Extensiones válidas
VALID_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.heic', '.webp'}

image_files = sorted([
    f for f in RAW_DIR.iterdir()
    if f.suffix.lower() in VALID_EXT
])

print(f'📁 Imágenes encontradas: {len(image_files)}\n')

if len(image_files) == 0:
    print('⚠️  No se encontraron imágenes en mis_imagenes/raw/')
    print('    Coloca tus fotos ahí con el formato: {digito}_{numero}.jpg')
else:
    for f in image_files:
        # Intentar extraer el label del nombre
        name = f.stem
        parts = name.split('_')
        label = parts[0] if parts[0].isdigit() else '?'
        print(f'   {f.name}  →  dígito: {label}')

---
## 3. Función de preprocesamiento

Convierte una foto de un dígito en papel al formato MNIST:
- Escala de grises
- Binarización con umbral adaptativo
- Recorte al dígito (bounding box)
- Redimensión a 20x20 con padding a 28x28 (como MNIST)
- Fondo negro, dígito blanco

In [ ]:
def preprocess_digit_photo(image_path, target_size=28):
    """
    Preprocesa una foto de un dígito escrito a mano para que sea
    compatible con el formato MNIST (28x28, fondo negro, dígito blanco).
    
    Returns: numpy array de shape (28, 28) con valores en [0, 255]
    """
    # Cargar y convertir a escala de grises
    img = Image.open(image_path).convert('L')
    
    # Redimensionar si es muy grande (para velocidad)
    max_dim = 500
    if max(img.size) > max_dim:
        img.thumbnail((max_dim, max_dim), Image.LANCZOS)
    
    # Convertir a array numpy
    img_array = np.array(img, dtype=np.float32)
    
    # Determinar si el fondo es claro u oscuro
    # Muestrear las esquinas para determinar el color del fondo
    h, w = img_array.shape
    corners = [
        img_array[:h//10, :w//10].mean(),      # esquina superior izquierda
        img_array[:h//10, -w//10:].mean(),      # esquina superior derecha
        img_array[-h//10:, :w//10].mean(),      # esquina inferior izquierda
        img_array[-h//10:, -w//10:].mean()      # esquina inferior derecha
    ]
    bg_color = np.mean(corners)
    
    # Si el fondo es claro (>128), invertir para que sea fondo negro
    if bg_color > 128:
        img_array = 255.0 - img_array
    
    # Binarizar con umbral de Otsu simplificado
    threshold = np.mean(img_array) + 0.5 * np.std(img_array)
    binary = (img_array > threshold).astype(np.float32) * 255
    
    # Encontrar bounding box del dígito
    rows = np.any(binary > 0, axis=1)
    cols = np.any(binary > 0, axis=0)
    
    if not rows.any():
        # Si no se encontró nada, retornar la imagen redimensionada
        img_resized = Image.fromarray(img_array.astype(np.uint8)).resize(
            (target_size, target_size), Image.LANCZOS)
        return np.array(img_resized)
    
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    
    # Agregar un pequeño margen
    margin = 5
    rmin = max(0, rmin - margin)
    rmax = min(binary.shape[0] - 1, rmax + margin)
    cmin = max(0, cmin - margin)
    cmax = min(binary.shape[1] - 1, cmax + margin)
    
    # Recortar al dígito
    cropped = img_array[rmin:rmax+1, cmin:cmax+1]
    
    # Redimensionar a 20x20 manteniendo aspecto (como MNIST)
    inner_size = 20
    h, w = cropped.shape
    if h > w:
        new_h = inner_size
        new_w = max(1, int(w * inner_size / h))
    else:
        new_w = inner_size
        new_h = max(1, int(h * inner_size / w))
    
    cropped_img = Image.fromarray(cropped.astype(np.uint8))
    resized = cropped_img.resize((new_w, new_h), Image.LANCZOS)
    resized_array = np.array(resized, dtype=np.float32)
    
    # Colocar en canvas de 28x28 centrado (con padding)
    canvas = np.zeros((target_size, target_size), dtype=np.float32)
    y_offset = (target_size - new_h) // 2
    x_offset = (target_size - new_w) // 2
    canvas[y_offset:y_offset+new_h, x_offset:x_offset+new_w] = resized_array
    
    return canvas.astype(np.uint8)

print('✅ Función de preprocesamiento definida')

---
## 4. Procesar todas las imágenes

In [ ]:
if len(image_files) == 0:
    print('⚠️  No hay imágenes para procesar. Agrega fotos a mis_imagenes/raw/')
else:
    processed = []
    
    for img_path in image_files:
        name = img_path.stem
        parts = name.split('_')
        
        if not parts[0].isdigit():
            print(f'⚠️  Saltando {img_path.name} — el nombre no empieza con un dígito')
            continue
        
        label = int(parts[0])
        
        try:
            result = preprocess_digit_photo(img_path)
            
            # Guardar imagen procesada
            out_path = PROC_DIR / f'{name}.png'
            Image.fromarray(result).save(out_path)
            
            processed.append({
                'filename': img_path.name,
                'label': label,
                'processed_path': str(out_path),
                'original_path': str(img_path)
            })
            
        except Exception as e:
            print(f'❌ Error procesando {img_path.name}: {e}')
    
    print(f'\n✅ Procesadas: {len(processed)} imágenes')
    
    # Guardar CSV con metadatos
    csv_path = PROC_DIR / 'labels.csv'
    with open(csv_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['filename', 'label', 'processed_path', 'original_path'])
        writer.writeheader()
        writer.writerows(processed)
    
    print(f'💾 Metadatos guardados en: {csv_path}')

---
## 5. Visualizar antes y después

In [ ]:
if len(image_files) == 0:
    print('⚠️  No hay imágenes para visualizar')
elif len(processed) > 0:
    n_images = len(processed)
    n_cols = 4  # pares de original + procesada
    n_rows = int(np.ceil(n_images / (n_cols // 2)))
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
    fig.suptitle('Antes y Después del Preprocesamiento', fontsize=22, fontweight='bold', y=1.01)
    
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    
    for i, item in enumerate(processed):
        row = i // (n_cols // 2)
        col = (i % (n_cols // 2)) * 2
        
        if row >= n_rows:
            break
        
        # Original
        orig = Image.open(item['original_path']).convert('L')
        axes[row, col].imshow(orig, cmap='gray')
        axes[row, col].set_title(f'Original: {item["filename"]}', fontsize=11)
        axes[row, col].axis('off')
        
        # Procesada
        proc = Image.open(item['processed_path'])
        axes[row, col+1].imshow(proc, cmap='gray')
        axes[row, col+1].set_title(f'Procesada (28x28) — Label: {item["label"]}', fontsize=11)
        axes[row, col+1].axis('off')
    
    # Ocultar ejes vacíos
    for i in range(len(processed), n_rows * (n_cols // 2)):
        row = i // (n_cols // 2)
        col = (i % (n_cols // 2)) * 2
        if row < n_rows:
            axes[row, col].axis('off')
            axes[row, col+1].axis('off')
    
    plt.tight_layout()
    plt.savefig('../resultados/antes_despues.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('💾 Guardado en resultados/antes_despues.png')

---
## 6. Resumen de imágenes procesadas

In [ ]:
if len(processed) > 0:
    # Grid de todas las imágenes procesadas
    n = len(processed)
    cols_grid = min(10, n)
    rows_grid = int(np.ceil(n / cols_grid))
    
    fig, axes = plt.subplots(rows_grid, cols_grid, figsize=(2 * cols_grid, 2.5 * rows_grid))
    fig.suptitle(f'Todas las imágenes procesadas ({n} total)', fontsize=20, fontweight='bold', y=1.02)
    
    if rows_grid == 1 and cols_grid == 1:
        axes = np.array([[axes]])
    elif rows_grid == 1:
        axes = axes.reshape(1, -1)
    elif cols_grid == 1:
        axes = axes.reshape(-1, 1)
    
    for i, item in enumerate(processed):
        r, c = i // cols_grid, i % cols_grid
        img = Image.open(item['processed_path'])
        axes[r, c].imshow(img, cmap='gray')
        axes[r, c].set_title(f'Label: {item["label"]}', fontsize=12, fontweight='bold')
        axes[r, c].axis('off')
    
    # Ocultar ejes vacíos
    for i in range(n, rows_grid * cols_grid):
        r, c = i // cols_grid, i % cols_grid
        axes[r, c].axis('off')
    
    plt.tight_layout()
    plt.savefig('../resultados/todas_procesadas.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Distribución por dígito
    labels = [item['label'] for item in processed]
    unique_labels, label_counts = np.unique(labels, return_counts=True)
    print(f'\n📊 Distribución de dígitos propios:')
    for label, count in zip(unique_labels, label_counts):
        print(f'   Dígito {label}: {count} imágenes')
    print(f'   Total: {len(processed)} imágenes')
    
    print(f'\n✅ ¡Listo! Ahora ejecuta el notebook 03 para probar las imágenes con el modelo.')
else:
    print('⚠️  No hay imágenes procesadas para mostrar')